# Experiment 2: OpenAI with GPT

## Setup

In [2]:
from os import getenv
from dotenv import load_dotenv, find_dotenv

def get_openai_api_key():
    _ = load_dotenv(find_dotenv())
    openai_api_key = getenv('OPENAI_API_KEY')
    return openai_api_key

llm_config = {
    'model': 'gpt-4o-mini',
    'api_key': get_openai_api_key(),
}

## Load documents

In [3]:
from PyPDF2 import PdfReader
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

def extract_text_from_pdf(filepath):
    with open(filepath, 'rb') as file:
        reader = PdfReader(file)
        text = '\n\n'.join([page.extract_text() for page in reader.pages if page.extract_text()])
    return text
    
docs = {
    'overview_statement': extract_text_from_pdf('../data/overview_statement.pdf'),
    'tasks': extract_text_from_pdf('../data/task_breakdown.pdf'),
}

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = InMemoryVectorStore(embedding=embedding_model)

for name, text in docs.items():
    doc = Document(page_content=text, metadata={'source': name})
    vector_store.add_documents([doc])

def query_rag_with_embedding(prompt, top_k=3, max_doc_length=1000):
    relevant_docs = vector_store.similarity_search(prompt, k=top_k)    
    truncated_docs = [doc.page_content[:max_doc_length] for doc in relevant_docs]
    context = '\n\n'.join(truncated_docs)
    return f'Using the following retrieved information, answer the question: {prompt}\n\n{context}'

# Define roles

In [4]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

# customer and manager
cp = \
    UserProxyAgent(
        name='CP',
        description='Customer Proxy',
        human_input_mode='NEVER',
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        code_execution_config={
            'last_n_messages': 1,
            'work_dir': 'tasks',
            'use_docker': False,
        },
    )
pm = \
    AssistantAgent(
        name='PM',
        description='Project Manager',
        llm_config=llm_config,
        system_message= \
            'You are a project manager, a natural leader who excels in team organization. ' + \
            'You monitor project progress, set deadlines and manage the budget. ' + \
            'You take immediate action to overcome obstacles using your creative problem-solving skills.',
    )

# engineers
re = \
    AssistantAgent(
        name='RE',
        description='Requirements Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a requirements engineer who analyzes software requirements. ' + \
            'You are responsible for creating a report according to the requirements. ' + \
            'Most importantly, you are to attend meetings with clients and project managers to collect these requirements.',
    )
se = \
    AssistantAgent(
        name='SE',
        description='Systems Engineer',
        llm_config=llm_config,
        system_message= \
            "You are a system engineer capable of architecting complex software systems that meet client's requirements. " + \
            'When the system is running, you are required to troubleshoot errors and patch security threats.',
    )
sd = \
    AssistantAgent(
        name='SD',
        description='Software Developer',
        llm_config=llm_config,
        system_message= \
            'You are a software developer tasked with implementing and maintaining software programs. ' + \
            'This activity involves producing a new source code and integrating existing software components. ' + \
            'However, the bulk of the testing is performed by a test engineer.',
    )
te = \
    AssistantAgent(
        name='TE',
        description='Test Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a test engineer, working closely with other engineers to identify software defects. ' + \
            'You are an expert in testing frameworks for executing test plans to ensure the software meets quality standards.',
    )
de = \
    AssistantAgent(
        name='DE',
        description='Documentation Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a documentation engineer who creates supporting documents to help others keep track of their work. ' + \
            'These documents often include visual graphics and technical manuals.',
    )

# chats
chat_manager = \
    GroupChatManager(
        groupchat= \
            GroupChat(
            agents=[cp, pm, re, se, sd, te, de],
            messages=[],
            speaker_selection_method='round_robin',
            allow_repeat_speaker=False,
            max_round=1,
        ),
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        llm_config=llm_config,
        code_execution_config={
            'work_dir': 'coding',
            'use_docker': False,
        },
    )

## Start chat

In [6]:
def any_message(*sections):
    message = f"Predict the amount of effort required to complete sections:\n\n"
    for _, section in enumerate(sections):
        message += f'- {section}\n'
    return message

pm.register_nested_chats(
    [
        {
            'recipient': chat_manager,
            'summary_method': 'reflection_with_llm',
            'clear_history': False,
        },
        {
            'recipient': pm,
            'message': any_message('Project plan', 'Risk Mitigation and Contingency Plan'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': re,
            'message': any_message('Requirement'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': se,
            'message': any_message('Analysis', 'Design'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': sd,
            'message': any_message('Coding and unit test'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': te,
            'message': any_message('Testing'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': de,
            'message': any_message('Documentation'),
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': chat_manager,
            'message': 'Summarize the effort estimation.',
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
    ],
    trigger=cp,
)
cp.initiate_chats([
    {
        'recipient': pm,
        'message': query_rag_with_embedding(
            'Consider the overview statement document of the project Chicago WideCase Smart-Home Services. ' + \
            'Given the work listed in the task description document\n\n.' + \
            '- Tag each task with a unique ID that starts with `REQ-001`.\n' + \
            '- Create effort estimate for each task.',
        ),
        'max_turns': 1,
        'summary_method': 'last_msg',
    }
])


********************************************************************************
Starting a new chat....

********************************************************************************
CP (to PM):

Using the following retrieved information, answer the question: Consider the overview statement document of the project Chicago WideCase Smart-Home Services. Given the work listed in the task description document

.- Tag each task with a unique ID that starts with `REQ-001`.
- Create effort estimate for each task.

 
Task  Amount of Work  Productivity  Rate  
Project Plan      
Write Plan  56 pages  5 page s/Hour  
Review Plan      
Preparation for review    4 pages/Hour  
Review Meeting   8 pages/Hour  
Rework  39 defects  5 defects/Hour  
   
Risk Mitigation and Contingency Plan      
Write Plan  78 pages  5 page s/Hour  
Review Plan      
Preparation for review    5 pages/Hour  
Review Meeting   10 pages/Hour  
Rework  19 defects  5 defects/Hour  
   
Requirement      
Write requiremen

[ChatResult(chat_id=None, chat_history=[{'content': 'Using the following retrieved information, answer the question: Consider the overview statement document of the project Chicago WideCase Smart-Home Services. Given the work listed in the task description document\n\n.- Tag each task with a unique ID that starts with `REQ-001`.\n- Create effort estimate for each task.\n\n \nTask  Amount of Work  Productivity  Rate  \nProject Plan      \nWrite Plan  56 pages  5 page s/Hour  \nReview Plan      \nPreparation for review    4 pages/Hour  \nReview Meeting   8 pages/Hour  \nRework  39 defects  5 defects/Hour  \n   \nRisk Mitigation and Contingency Plan      \nWrite Plan  78 pages  5 page s/Hour  \nReview Plan      \nPreparation for review    5 pages/Hour  \nReview Meeting   10 pages/Hour  \nRework  19 defects  5 defects/Hour  \n   \nRequirement      \nWrite requirements  176 Req 4 Req/Hour  \nWrite Use Case Model  62 Use Cases  5 use case/ 2 Hour s \nReview Requirements / Use Case Model     